# 06 · Recover tissue regions from impure images
**Paper:** Steps 5 – 7 · Fig. 2C

| Step | Rule |
|---|---|
| **5 · Candidate scoring** | keep *Pathology Tissue* boxes with confidence ≥ 0.5. Single-panel image → keep the box with the largest **confidence × area** |
| **6 · Multi-panel split** | images flagged *Multi-Panel* by QuiltCleaner → visit boxes in descending confidence × area and keep each one unless its IoU with a kept box is > 0.5. Every kept box is one panel |
| **7 · Filtering & assembly** | drop crops whose width **or** height is < 224 px. Each crop inherits the source caption. Clean images + crops form the final dataset |

The rules live in [`requilt/recovery.py`](../requilt/recovery.py) and
[`requilt/boxes.py`](../requilt/boxes.py), so this notebook only wires them together.

**Outputs** → `outputs/recovered/`, `outputs/recovered_metadata.csv`,
`outputs/final_dataset/`, `outputs/final_dataset.csv`

In [ ]:
# --- setup: make the `requilt` package importable and load the shared config ---
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # repo root (not needed after `pip install -e .`)

from requilt import load_config

cfg = load_config()      # configs/config.yaml
P = cfg.paths            # all paths, already resolved to absolute Paths

In [ ]:
import shutil

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from ultralytics import YOLO

from requilt.data import MULTI_PANEL, load_caption_map
from requilt.recovery import METADATA_COLUMNS, RecoveryStats, recover_image

R = cfg.recovery
RECOVERED_DIR = P.output_dir / "recovered"
RECOVERED_DIR.mkdir(parents=True, exist_ok=True)

## 1 · Load the detector, the multi-panel flags and the captions

In [ ]:
model = YOLO(P.weights)
TISSUE = {name: idx for idx, name in model.names.items()}[R.target_class]
print(f"'{R.target_class}' -> class index {TISSUE}")

test_dirty = pd.read_csv(P.evaluation_dir / "test_dirty_sample.csv")
test_clean = pd.read_csv(P.evaluation_dir / "test_clean_sample.csv")
is_multipanel = dict(zip(test_dirty["Filename"], test_dirty[MULTI_PANEL] >= cfg.impurity.threshold))
captions = load_caption_map(P.caption_lookup_csv)

print(f"impure evaluation images: {len(test_dirty)}  ({sum(is_multipanel.values())} multi-panel)")

## 2 · Detect and crop (Steps 5 – 7)
Inference uses a low confidence pre-filter (0.1). The 0.5 threshold of Step 5 is applied
inside `recover_image`.

In [ ]:
stats, rows = RecoveryStats(), []
for name in test_dirty["Filename"]:
    # one image at a time: passing a whole list to predict() loads every image into memory
    result = model.predict(source=str(P.quilt_images_dir / name), imgsz=cfg.train.img_size,
                           conf=R.predict_conf, verbose=False)[0]
    rows += recover_image(
        result,
        class_index=TISSUE,
        is_multipanel=is_multipanel[name],
        caption=captions.get(name),
        out_dir=RECOVERED_DIR,
        cfg_recovery=R,
        stats=stats,
    )

recovered = pd.DataFrame(rows, columns=METADATA_COLUMNS)
recovered.to_csv(P.output_dir / "recovered_metadata.csv", index=False)
print(stats.summary())

## 3 · Assemble the final dataset (Step 7)
Clean images are copied unchanged and recovered crops are added next to them. Each item
keeps the caption of its source image.

In [ ]:
FINAL_DIR = P.output_dir / "final_dataset"
FINAL_DIR.mkdir(exist_ok=True)

clean_rows = []
for name in test_clean["Filename"]:
    shutil.copy2(P.quilt_images_dir / name, FINAL_DIR / name)
    clean_rows.append({"crop_filename": name, "source_filename": name,
                       "caption": captions.get(name), "is_multipanel": False})

for name in recovered["crop_filename"]:
    shutil.copy2(RECOVERED_DIR / name, FINAL_DIR / name)

final = pd.concat([pd.DataFrame(clean_rows), recovered], ignore_index=True)
final.to_csv(P.output_dir / "final_dataset.csv", index=False)

print(f"clean images    : {len(clean_rows)}")
print(f"recovered crops : {len(recovered)}") 
print(f"total samples   : {len(final)}")     
print(f"missing captions: {final['caption'].isna().sum()}")